# Preparación y exploración
Ulises González Coronado. Energy Efficiency, UCI.

Pregunta: ¿cómo cambian las cargas según compacidad y altura? Primero revisamos el archivo y después generamos sus derivados. El original se conserva intacto.

In [1]:
from pathlib import Path
import sys
import pandas as pd
ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print("Carpeta del proyecto:", ROOT.name)


Carpeta del proyecto: REC_ECBD_2026_Gonzalez_Ulises


## 1. Revisar el archivo original
X1-X8 son características de entrada; Y1 y Y2 son las cargas de calefacción y refrigeración.

In [2]:
bruto = pd.read_excel(ROOT / "data/raw/ENB2012_data.xlsx")
print("Filas y columnas:", bruto.shape)
print(bruto.head().to_string(index=False))
print("Faltantes:", int(bruto.isna().sum().sum()))
print("Duplicados:", int(bruto.duplicated().sum()))

Filas y columnas: (768, 10)
  X1    X2    X3     X4  X5  X6  X7  X8    Y1    Y2
0.98 514.5 294.0 110.25 7.0   2 0.0   0 15.55 21.33
0.98 514.5 294.0 110.25 7.0   3 0.0   0 15.55 21.33
0.98 514.5 294.0 110.25 7.0   4 0.0   0 15.55 21.33
0.98 514.5 294.0 110.25 7.0   5 0.0   0 15.55 21.33
0.90 563.5 318.5 122.50 7.0   2 0.0   0 20.84 28.28
Faltantes: 0
Duplicados: 0


## 2. Ejecutar la preparación
La función valida tipos, rangos, categorías y duplicados. Añade columnas de apoyo y construye el Data Mart local. Su código comentado está en src/preparacion_datos.py.

In [3]:
from src.preparacion_datos import preparar
datos, calidad = preparar()
print("Firma SHA-256 del original:", calidad["sha256_original"])

Preparación terminada: 768 registros, 8 entradas y 2 objetivos.
Faltantes: 0. Duplicados: 0. No se eliminaron ni imputaron datos.
Compacidad y altura: no existen compacidades compartidas por ambas alturas.
SQLite complementario validado. Para el modelo estrella en SQL Server: python -m src.etl_sqlserver --cargar
Firma SHA-256 del original: 0089fcffc1415e41e2ff63730cca5280efe54fc43d722dfde1b0aaa808e35dc4


## 3. Comparar grupos
Cada valor de compacidad aparece en una sola altura. Las diferencias describen configuraciones completas; no identifican el efecto aislado de una variable.

In [4]:
resumen = pd.read_csv(ROOT / "data/processed/resumen_compacidad_altura.csv")
print(resumen.round(3).to_string(index=False))
print("Alturas por compacidad:")
print(datos.groupby("X1").X5.nunique().to_string())

  X1  X5  registros  calefaccion_media  calefaccion_mediana  calefaccion_desviacion  refrigeracion_media  refrigeracion_mediana  refrigeracion_desviacion
0.62 3.5         64             14.283               14.335                   2.186               15.244                 15.105                     1.384
0.64 3.5         64             16.615               16.850                   2.115               20.227                 20.200                     1.409
0.66 3.5         64             12.817               12.930                   2.092               15.870                 15.920                     1.745
0.69 3.5         64             12.389               12.570                   1.960               15.242                 15.295                     1.407
0.71 3.5         64             12.040               12.280                   2.040               15.035                 14.955                     1.503
0.74 3.5         64             11.887               12.140                 

![Cargas por compacidad y altura](../docs/figuras/01_compacidad_altura.png)

![Distribución por altura](../docs/figuras/02_cargas_altura.png)

Las cargas conservan la escala original: la ficha UCI consultada no declara sus unidades.

## 4. Consultar el Data Mart
SQLite permite comprobar localmente la carga. Los scripts de SQL Server todavía deben ejecutarse en ese motor.

In [5]:
import sqlite3
with sqlite3.connect((ROOT / "data/processed/datamart_energia.sqlite").as_uri() + "?mode=ro", uri=True) as conexion:
    consulta = (ROOT / "sql/03_consultar_cargas.sql").read_text(encoding="utf-8")
    print(pd.read_sql_query(consulta, conexion).round(3).to_string(index=False))
    print("Errores de llaves foráneas:", conexion.execute("PRAGMA foreign_key_check").fetchall())

  X1  X5  registros  calefaccion_media  refrigeracion_media
0.62 3.5         64             14.283               15.244
0.64 3.5         64             16.615               20.227
0.66 3.5         64             12.817               15.870
0.69 3.5         64             12.389               15.242
0.71 3.5         64             12.040               15.035
0.74 3.5         64             11.887               14.810
0.76 7.0         64             35.664               36.407
0.79 7.0         64             38.608               40.239
0.82 7.0         64             25.555               28.028
0.86 7.0         64             28.553               30.912
0.90 7.0         64             31.629               33.821
0.98 7.0         64             27.646               29.217
Errores de llaves foráneas: []


Fuente: Tsanas y Xifara (2012), https://doi.org/10.24432/C51307. Datos simulados. Preparación y resultados obtenidos mediante las funciones del proyecto.